## 1. Imports

In [1]:
# ── Standard Library ──────────────────────────────────────────────────────────
import sys, time, ast, json, re, os
from datetime import datetime, timedelta, date
from itertools import combinations

# ── Data ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from tqdm import tqdm

# ── Stats ─────────────────────────────────────────────────────────────────────
from scipy.stats import ttest_ind

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ── Boosting ──────────────────────────────────────────────────────────────────
import lightgbm as lgb
import joblib


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config

In [3]:
# ── Mode: always live ─────────────────────────────────────────────────────────
MODE  = 'live'
TODAY = str(date.today())

# ── Paths (local — files downloaded from Google Drive by workflow) ─────────────
BASE_PATH  = "/content/drive/MyDrive/PT" if os.path.exists("/content/drive/MyDrive/PT") else "."

PATHS = {
    'runners':     f'{BASE_PATH}/runners.parquet',
    'webTips':     f'{BASE_PATH}/webTips.parquet',
    'odds':        f'{BASE_PATH}/odds.parquet',
    'races':       f'{BASE_PATH}/races.parquet',
    'runners_tdy': f'{BASE_PATH}/runners_tdy.parquet',
    'webTips_tdy': f'{BASE_PATH}/webTips_tdy.parquet',
    'odds_tdy':    f'{BASE_PATH}/odds_tdy.parquet',
    'races_tdy':   f'{BASE_PATH}/races_tdy.parquet',
    'tips_out':    f'{BASE_PATH}/today_tips.parquet',
    'model':       f'{BASE_PATH}/pt_model.pkl',
    'scaler':      f'{BASE_PATH}/pt_scaler.pkl',
    'features':    f'{BASE_PATH}/pt_features.pkl',
}

# ── Market filter ─────────────────────────────────────────────────────────────
ODDS_SUM_MIN, ODDS_SUM_MAX = 1.1, 1.4

# ── Place-market overround ─────────────────────────────────────────────────────
PLACE_MARGIN  = 1.2
WIN_OVERROUND = 1.175

print(f'Mode: {MODE}  |  Today: {TODAY}')

Mode: live  |  Today: 2026-04-22


## 3. Load Saved Model

In [4]:
model        = joblib.load(PATHS['model'])
global_scaler = joblib.load(PATHS['scaler'])
ALL_FEATURES  = joblib.load(PATHS['features'])

print(f'✅ Model loaded  |  Features: {len(ALL_FEATURES)}')


✅ Model loaded  |  Features: 85


## 4. Load Data

In [5]:
def load_parquet(path, dedup_cols=None):
    df = pd.read_parquet(path)
    if dedup_cols:
        df = df.drop_duplicates(subset=dedup_cols)
    else:
        df = df.drop_duplicates()
    return df

# ── Historical data ───────────────────────────────────────────────────────────
runners  = load_parquet(PATHS['runners'],  dedup_cols=['horseName', 'raceId'])
webTips  = load_parquet(PATHS['webTips'],  dedup_cols=['meetingId', 'raceId'])
webTips  = webTips[webTips['text'].notna()]
odds     = load_parquet(PATHS['odds'])
races    = load_parquet(PATHS['races'])

# ── Today's data ──────────────────────────────────────────────────────────────
runners_tdy  = load_parquet(PATHS['runners_tdy'],  dedup_cols=['horseName', 'raceId'])
webTips_tdy  = load_parquet(PATHS['webTips_tdy'],  dedup_cols=['meetingId', 'raceId'])
odds_tdy     = load_parquet(PATHS['odds_tdy'])
races_tdy    = load_parquet(PATHS['races_tdy'])

print(f'runners: {runners.shape}  |  runners_tdy: {runners_tdy.shape}')

runners: (315038, 59)  |  runners_tdy: (129, 59)


## 5. Going Overrides

In [6]:
import json
from datetime import date

if os.path.exists(f"{BASE_PATH}/going_overrides.json"):
    with open(f"{BASE_PATH}/going_overrides.json") as f:
        overrides = json.load(f)

    override_date = overrides.get('date', '')
    today_str = str(date.today())

    if override_date == today_str:
        # Apply meeting-level overrides
        for meeting, going in overrides.get('by_meeting', {}).items():
            races_tdy.loc[races_tdy['name_meeting'] == meeting, 'going'] = going

        # Apply race-name overrides (more specific, applied after)
        for race_name, going in overrides.get('by_race_name', {}).items():
            races_tdy.loc[races_tdy['name_race'] == race_name, 'going'] = going

        print(f"✅ Going overrides applied for {today_str}")
    else:
        print(f"⚠️ going_overrides.json is dated '{override_date}', today is '{today_str}' — skipping overrides")
else:
    print("⚠️ No going_overrides.json found — no overrides applied")

✅ Going overrides applied for 2026-04-22


## 6. Merge

In [7]:
RACE_COLS = ['id_race', 'class', 'type', 'going', 'date', 'totalPrize',
             'distance', 'name_meeting', 'direction', 'rail', 'surface']

def merge_all(runners_df, races_df, odds_df, webTips_df):
    df = runners_df.merge(races_df[RACE_COLS],
                          left_on='raceId', right_on='id_race', how='left')
    df = df.merge(odds_df[['raceId', 'horseName', 'referenceOdd', 'liveOdd']],
                  on=['raceId', 'horseName'], how='left')
    df = df.merge(webTips_df[['raceId', 'meetingId', 'text']],
                  on=['raceId', 'meetingId'], how='left')
    return df

runners     = merge_all(runners,     races,     odds,     webTips)
runners_tdy = merge_all(runners_tdy, races_tdy, odds_tdy, webTips_tdy)
runners = runners[runners['ranking'].isnull()==False]
print(f'Merged runners: {runners.shape}  |  Today: {runners_tdy.shape}')

runners = runners[runners['ranking'].isnull()==False]

Merged runners: (302714, 73)  |  Today: (129, 73)


## 7. Combine Historical + Today

In [8]:
runners = pd.concat([runners, runners_tdy], ignore_index=True)
print(f'Combined runners: {runners.shape}')


Combined runners: (302843, 73)


/tmp/ipykernel_10878/184532812.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  runners = pd.concat([runners, runners_tdy], ignore_index=True)


## 8. Margin → Lengths

In [9]:
MARGIN_MAP = {
    'DH': 0, 'NEZ': 0.05, 'CTT': 0.1,
    'TETE': 0.2, 'CTE': 0.3, 'ENC': 0.4, 'LOIN': 99
}

def parse_margin(margin) -> float:
    if isinstance(margin, str):
        m = margin.replace('(', '').replace(')', '').strip()
        if m in MARGIN_MAP:
            return MARGIN_MAP[m]
        total = 0.0
        for part in m.split():
            if '/' in part:
                num, denom = part.split('/')
                total += float(num) / float(denom)
            else:
                try:
                    total += float(part)
                except ValueError:
                    pass
        return total
    if pd.isna(margin):
        return 0.0
    return float(margin)

def build_margin_lookup(df):
    return {(row.raceId, row.ranking): row.margin for row in df.itertuples(index=False)}

def margin_to_lengths(margin, ranking, raceId, lookup) -> float:
    if ranking == 1:
        margin_r2 = lookup.get((raceId, 2))
        if margin_r2 is None:
            return 0.0
        return -parse_margin(margin_r2)
    return parse_margin(margin)

# In live mode: keep today's rows even without ranking
runners_hist = runners[runners['ranking'].notna()]
margin_lookup = build_margin_lookup(runners_hist)

runners['lengths_back'] = runners.apply(
    lambda r: margin_to_lengths(r['margin'], r['ranking'], r['raceId'], margin_lookup)
    if pd.notna(r['ranking']) else np.nan,
    axis=1
)
runners['cumulative_lengths_back'] = (
    runners[runners['lengths_back'] >= 0]
    .sort_values('ranking')
    .groupby('raceId')['lengths_back']
    .cumsum()
)
runners['cumulative_lengths_back'] = runners['cumulative_lengths_back'].fillna(runners['lengths_back'])
print('lengths_back computed')


lengths_back computed


## 9. Basic Features

In [10]:
runners['runners'] = runners.groupby('raceId')['horseId'].transform('count')
runners['pos_perc'] = (runners['runners'] - runners['ranking']) / (runners['runners'] - 1)
runners['place']    = np.where(
    (runners['ranking'] <= 2) | ((runners['ranking'] == 3) & (runners['runners'] >= 7)), 1, 0
)
runners['win'] = (runners['ranking'] == 1).astype(int)


## 10. Going Category & Distance Group

In [11]:
GOING_MAP = {
    'Lourd': 'VERY SLOW', 'Très lourd': 'VERY SLOW', 'Collant': 'VERY SLOW',
    'Souple': 'SLOW',     'Très souple': 'SLOW',
    'Bon souple': 'FAST', 'Bon': 'FAST',
    'Léger': 'VERY FAST', 'Bon léger': 'VERY FAST', 'Très léger': 'VERY FAST',
    'PSF Standard': 'PSF', 'PSF Lente': 'PSF', 'PSF Rapide': 'PSF',
}

def distance_group(m):
    if pd.isna(m):
        return None
    m = int(m)
    if m <= 1000:  return '0-1000'
    if m > 3600:   return '>3600'
    lo = ((m - 1) // 200) * 200 + 1
    return f'{lo}-{lo + 199}'

runners['going_category'] = runners['going'].map(GOING_MAP)
runners['distance_group'] = runners['distance'].apply(distance_group)
runners['date']            = pd.to_datetime(runners['date'])

if 'meetingName' not in runners.columns and 'name_meeting' in runners.columns:
    runners['meetingName'] = runners['name_meeting']

print('going_category & distance_group done')


going_category & distance_group done


## 11. Draw & Prize Money

In [12]:
runners['draw_unique'] = (
    runners[['draw', 'name_meeting', 'distance', 'direction', 'rail', 'surface']]
    .apply(lambda r: ' - '.join(str(x) for x in r if pd.notna(x)), axis=1)
)

PRIZE_SHARES = {1: 0.50, 2: 0.20, 3: 0.10, 4: 0.05}

def prize_money(row):
    share = PRIZE_SHARES.get(int(row['ranking']), 0.0) if pd.notna(row['ranking']) else 0.0
    return share * row.get('totalPrize_y', row.get('totalPrize', 0))

runners['prizemoney'] = runners.apply(prize_money, axis=1)
runners['handicapRating_adj'] = runners['handicapRatingKg'] + 0.625 * (55 - runners['weightKg'])


## 12. Rolling Stats

In [13]:
stats_predictors = []

def add_rolling_stats(df, entity_col, windows, metrics=('prizemoney', 'pos_perc'), suffix_map=None):
    for window in windows:
        suffix = suffix_map.get(window, window) if suffix_map else window
        stat_base = (
            df.groupby(['date', entity_col])[list(metrics)]
            .mean()
            .reset_index()
        )
        rolled = (
            stat_base.set_index('date')
            .groupby(entity_col)
            .rolling(window, closed='left')[list(metrics)]
            .mean()
            .reset_index()
            .rename(columns={m: f'{entity_col}_{m}{suffix}' for m in metrics})
        )
        df = df.merge(rolled, on=['date', entity_col], how='left')
        for m in metrics:
            col = f'{entity_col}_{m}{suffix}'
            if col not in stats_predictors:
                stats_predictors.append(col)
    return df

print('horseSir (9999D)...')
runners = add_rolling_stats(runners, 'horseSir', ['9999D'])

print('draw_unique (9999D)...')
stat_du = runners.groupby(['date', 'draw_unique'])[['pos_perc']].mean().reset_index()
rolled_du = (
    stat_du.set_index('date')
    .groupby('draw_unique')
    .rolling('9999D', closed='left')[['pos_perc']]
    .mean()
    .reset_index()
    .rename(columns={'pos_perc': 'draw_unique_posperc9999D'})
)
runners = runners.merge(rolled_du, on=['date', 'draw_unique'], how='left')
stats_predictors.append('draw_unique_posperc9999D')

for entity in ['trainerName', 'jockeyName', 'horseName', 'ownerName']:
    print(f'{entity} (30D, 365D)...')
    runners = add_rolling_stats(runners, entity, ['30D', '365D'])

print(f'Rolling stats done — {len(stats_predictors)} predictors.')


horseSir (9999D)...
draw_unique (9999D)...
trainerName (30D, 365D)...
jockeyName (30D, 365D)...
horseName (30D, 365D)...
ownerName (30D, 365D)...
Rolling stats done — 19 predictors.


## 13. Race-Level Percentile Features

In [14]:
for entity in ['trainerName', 'horseName']:
    col_src = f'{entity}_prizemoney365D'
    col_out = f'{entity}_prizemoney365D_race_pct'
    race_mean = (
        runners.groupby('raceId')[col_src]
        .mean()
        .reset_index()
        .rename(columns={col_src: col_out})
    )
    runners = runners.merge(race_mean, on='raceId', how='left')
    stats_predictors.append(col_out)

print('Race-level percentile features done.')


Race-level percentile features done.


## 14. Context T-Scores

In [15]:
stats_predictors_gtd = []

CONTEXT_COMBOS = {
    'trainerName': ['name_meeting', 'jockeyName', 'type'],
    'jockeyName':  ['name_meeting', 'trainerName', 'going_category'],
    'horseId':     ['name_meeting', 'trainerName', 'going_category', 'jockeyName', 'distance_group'],
    'horseSir':    ['going_category', 'distance_group'],
}
THRESHOLD_MAP = {'trainerName': 10, 'jockeyName': 10, 'horseId': 3, 'horseSir': 10}

print('Calculating overall entity rolling stats...')
for entity in CONTEXT_COMBOS:
    window = '9999D' if entity == 'horseSir' else '10950D'
    overall = (
        runners.groupby(['date', entity])
        .agg(run=('ranking', 'count'), pos_perc_sum=('pos_perc', 'sum'))
        .reset_index()
    )
    overall_rolled = (
        overall.set_index('date')
        .groupby(entity)
        .rolling(window, closed='left')
        .agg({'pos_perc_sum': 'sum', 'run': 'sum'})
        .reset_index()
    )
    col_overall     = f'{entity}_pos_perc{window}'
    col_overall_run = f'{entity}_run{window}'
    overall_rolled[col_overall]     = overall_rolled['pos_perc_sum'] / overall_rolled['run']
    overall_rolled[col_overall_run] = overall_rolled['run']
    overall_rolled = overall_rolled[['date', entity, col_overall, col_overall_run]]
    runners = runners.drop(columns=[c for c in [col_overall, col_overall_run] if c in runners.columns])
    runners = runners.merge(overall_rolled, on=['date', entity], how='left')

print('Calculating context-specific rolling stats and t-scores...')
for entity, contexts in CONTEXT_COMBOS.items():
    window    = '9999D' if entity == 'horseSir' else '10950D'
    threshold = THRESHOLD_MAP[entity]
    col_overall     = f'{entity}_pos_perc{window}'
    col_overall_run = f'{entity}_run{window}'

    for ctx in contexts:
        print(f'  {entity} x {ctx}  ({window})')
        col_specific = f'{entity}{ctx}_pos_perc{window}'
        col_runs     = f'{entity}{ctx}_run{window}'

        spec = (
            runners.groupby(['date', entity, ctx])
            .agg(run=('ranking', 'count'), pos_perc_sum=('pos_perc', 'sum'))
            .reset_index()
        )
        spec_rolled = (
            spec.set_index('date')
            .groupby([entity, ctx])
            .rolling(window, closed='left')
            .agg({'pos_perc_sum': 'sum', 'run': 'sum'})
            .reset_index()
            .rename(columns={'run': col_runs})
        )
        spec_rolled[col_specific] = spec_rolled['pos_perc_sum'] / spec_rolled[col_runs]
        spec_rolled = spec_rolled[['date', entity, ctx, col_specific, col_runs]]
        runners = runners.drop(columns=[c for c in [col_specific, col_runs] if c in runners.columns])
        runners = runners.merge(spec_rolled, on=['date', entity, ctx], how='left')

        col_tscore = f'{entity}{ctx}_tscore{window}'
        runners[col_tscore] = np.where(
            runners[col_runs] >= threshold,
            (runners[col_specific] - runners[col_overall]) * np.sqrt(runners[col_runs]),
            np.nan
        )
        stats_predictors_gtd.append(col_tscore)

print(f'Context preference features done — {len(stats_predictors_gtd)} features.')


Calculating overall entity rolling stats...
Calculating context-specific rolling stats and t-scores...
  trainerName x name_meeting  (10950D)
  trainerName x jockeyName  (10950D)
  trainerName x type  (10950D)
  jockeyName x name_meeting  (10950D)
  jockeyName x trainerName  (10950D)
  jockeyName x going_category  (10950D)
  horseId x name_meeting  (10950D)
  horseId x trainerName  (10950D)
  horseId x going_category  (10950D)
  horseId x jockeyName  (10950D)
  horseId x distance_group  (10950D)
  horseSir x going_category  (9999D)
  horseSir x distance_group  (9999D)
Context preference features done — 13 features.


## 15. Last-Time-Out Features

In [16]:
runners = runners.sort_values('date', ascending=True).reset_index(drop=True)
runners['horse_run'] = runners.groupby('horseId').cumcount()

LTO_BASE_COLS = [
    'cumulative_lengths_back', 'pos_perc', 'prizemoney',
    'liveOdd', 'totalPrize_y', 'type', 'date',
]
LTO_LAGS     = [1, 2, 3]
LTO_CONTEXTS = ['going_category', 'distance_group', 'name_meeting']
LTO_EXCLUDE  = {'type', 'date'}

lto_predictors = []

for col in LTO_BASE_COLS:
    for lag in LTO_LAGS:
        tmp = runners[['horseId', 'horse_run', col]].copy()
        tmp['horse_run'] = tmp['horse_run'] + lag
        runners = runners.merge(
            tmp, on=['horseId', 'horse_run'], how='left', suffixes=['', f'_{lag}lto']
        )
        new_col = f'{col}_{lag}lto'
        if col not in LTO_EXCLUDE:
            lto_predictors.append(new_col)

for cat in LTO_CONTEXTS:
    for col in LTO_BASE_COLS:
        tmp = runners[['horseId', 'horse_run', cat, col]].copy()
        tmp['horse_run'] = tmp['horse_run'] + 1
        runners = runners.merge(
            tmp, on=['horseId', 'horse_run', cat], how='left',
            suffixes=['', f'_1lto_{cat}']
        )
        new_col = f'{col}_1lto_{cat}'
        if col not in LTO_EXCLUDE:
            lto_predictors.append(new_col)

for cat_a, cat_b in combinations(LTO_CONTEXTS, 2):
    combo = f'{cat_a}_{cat_b}'
    for col in ['pos_perc', 'cumulative_lengths_back']:
        tmp = runners[['horseId', 'horse_run', cat_a, cat_b, col]].copy()
        tmp['horse_run'] = tmp['horse_run'] + 1
        runners = runners.merge(
            tmp, on=['horseId', 'horse_run', cat_a, cat_b], how='left',
            suffixes=['', f'_1lto_{combo}']
        )
        lto_predictors.append(f'{col}_1lto_{combo}')

runners['datesince'] = (runners['date'] - runners['date_1lto']).dt.days

print(f'LTO features done — {len(lto_predictors)} predictors.')


LTO features done — 36 predictors.


## 16. WebTips Ranking

In [ ]:
def process_webtips_fast(webTips_df, runners_df):
    runners_df = runners_df.copy()
    runners_df['webTipRank']      = np.nan
    runners_df['webTipRank_perc'] = np.nan

    tip_lookup = {}
    for _, tip_row in webTips_df.iterrows():
        if pd.isna(tip_row.get('text')):
            continue
        rid = tip_row['raceId']
        if rid in tip_lookup:
            continue
        tip_lookup[rid] = tip_row['text']

    records = []
    race_groups = runners_df.groupby('raceId')['horseName'].apply(list)

    for rid, text in tqdm(tip_lookup.items(), desc='webTips'):
        if rid not in race_groups.index:
            continue
        horse_names = race_groups[rid]
        lower       = text.lower()
        positions = []
        for horse in horse_names:
            match = re.search(r'' + re.escape(horse.lower()) + r'', lower)
            if match:
                positions.append((match.start(), horse))
        positions.sort()
        n = len(positions)
        for rank, (_, horse) in enumerate(positions, start=1):
            records.append({
                'raceId':         rid,
                'horseName':      horse,
                'webTipRank':     rank,
                'webTipRank_perc': (n - rank) / (n - 1) if n > 1 else 0.5,
            })

    if not records:
        return runners_df

    ranks_df = pd.DataFrame(records)
    runners_df = runners_df.drop(columns=['webTipRank', 'webTipRank_perc'])
    runners_df = runners_df.merge(ranks_df, on=['raceId', 'horseName'], how='left')
    return runners_df

runners = process_webtips_fast(webTips, runners)
print('webTipRank_perc computed.')


webTips:  56%|█████▌    | 13497/24152 [00:05<00:06, 1706.56it/s]

## 17. Odds Features

In [ ]:
runners['odds']       = 1 / runners['liveOdd']
runners['odds_sum']   = runners.groupby(['raceId', 'name_meeting'])['odds'].transform('sum')
runners['place_sp']   = (runners['liveOdd'] - 1) / 4 + 1
runners['odds_place'] = 1 / runners['place_sp']

n_places       = runners.groupby('raceId')['place'].transform('sum')
sum_place_odds = runners.groupby('raceId')['odds_place'].transform('sum')
runners['odds_place'] *= (n_places * PLACE_MARGIN) / sum_place_odds

runners = runners.merge(
    races[['id_race', 'minAge']], left_on='raceId', right_on='id_race', how='left'
)
print('Odds features done.')


## 18. Feature Lists & Dataset

In [ ]:
TYPE_LTO_COLS   = [c for c in lto_predictors if 'type_' not in c]
predictors      = stats_predictors + TYPE_LTO_COLS + ['weightKg', 'handicapRating_adj'] + stats_predictors_gtd
predictors_cat  = [
    'going_category', 'distance_group', 'type', 'sex', 'blinkers', 'tongueTie',
    'type_1lto', 'type_2lto', 'type_3lto',
]
predictors_misc = ['runners', 'age', 'datesince', 'webTipRank_perc']
predictors_ungroup = predictors_cat + predictors_misc

# Live: keep today's rows
df = runners[runners['date'] >= TODAY].copy()

print(f'Dataset size: {df.shape}')


## 19. Encode Categoricals

In [ ]:
le = LabelEncoder()
for col in predictors_cat:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))

df.replace([np.inf, -np.inf], np.nan, inplace=True)


## 20. Group-wise Scaling

In [ ]:
def groupwise_scale(df, group_col, feature_cols):
    scaled_cols = [c + '_scaled' for c in feature_cols]
    result = pd.DataFrame(np.nan, index=df.index, columns=scaled_cols)

    for _, group in tqdm(df.groupby(group_col), desc='Scaling groups'):
        arr = group[feature_cols].values.astype(np.float64)
        out = np.full_like(arr, np.nan)
        for j in range(arr.shape[1]):
            col_vals = arr[:, j]
            mask     = ~np.isnan(col_vals)
            if mask.sum() > 1:
                scaler = StandardScaler()
                out[mask, j] = scaler.fit_transform(col_vals[mask].reshape(-1, 1)).flatten()
        result.loc[group.index, :] = out

    return result.fillna(0)

df_scaled = groupwise_scale(df, 'raceId', predictors)
df = pd.concat([df, df_scaled], axis=1)
scaled_cols = [c + '_scaled' for c in predictors]
print(f'Scaling done. {len(scaled_cols)} scaled columns added.')


## 21. Predict

In [ ]:
X = df[ALL_FEATURES + ['raceId', 'odds_sum', 'date']]
y = df[['place']]

X_full_final = global_scaler.transform(X[ALL_FEATURES].fillna(0))

label_preds = model.predict_proba(X_full_final)[:, 1]
print(f'✅ Predictions generated: {len(label_preds)}')


## 22. Build Output

In [ ]:
def build_evaluation_df(df, y_true, preds):
    eval_df = pd.DataFrame(y_true).copy()
    eval_df = eval_df.assign(
        place_sp        = df['place_sp'],
        win_sp          = df['liveOdd'],
        win_odds        = 1 / df['liveOdd'],
        horse           = df['horseName'],
        course          = df['name_meeting'],
        distype         = df['distance_group'],
        webTipRank_perc = df['webTipRank_perc'],
        going_category  = df['going_category'],
        win             = df['win'],
        age             = df['age'],
        place           = df['place'],
        position        = df['ranking'],
        racetype        = df['type'],
        date            = df['date'],
        raceId          = df['raceId'],
        y_pred_raw      = preds,
    )

    eval_df['y_pred'] = eval_df.groupby('raceId')['y_pred_raw'].transform(lambda x: x / x.sum())
    race_sum           = eval_df.groupby('raceId')['y_pred'].transform('sum')
    eval_df['adj_prob'] = eval_df['y_pred'] * (WIN_OVERROUND / race_sum)
    eval_df['SP']       = np.round(1 / eval_df['adj_prob'], 1)

    eval_df['runners']   = eval_df.groupby('raceId')['horse'].transform('count')
    eval_df['rank']      = eval_df.groupby('raceId')['y_pred'].rank(method='first', ascending=False)
    eval_df['rank_pct']  = (eval_df['runners'] - eval_df['rank']) / (eval_df['runners'] - 1)
    eval_df['is_top_pick'] = (eval_df['rank_pct'] == 1).astype(int)

    return eval_df

y_eval = build_evaluation_df(df, y, label_preds)
print(f'y_eval shape: {y_eval.shape}')

## 23. Save Today's Tips

In [ ]:
today_tips = (
    y_eval[y_eval['date'] == TODAY]
    .sort_values('SP')
    [['raceId', 'date', 'course', 'racetype', 'horse', 'distype', 'position', 'win_sp', 'is_top_pick', 'SP']]
)

today_tips.to_parquet(PATHS['tips_out'], engine='pyarrow', index=False)
print(f'✅ today_tips saved → {PATHS["tips_out"]}  ({len(today_tips)} rows)')
today_tips
